# Lecture 06 solutions — Clustering

Complete runnable worked answers.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "data" / "customer_segments.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/customer_segments.csv")


ROOT = repo_root()
customers = pd.read_csv(ROOT / "data" / "customer_segments.csv")
features = [
    "recency_days",
    "orders_12m",
    "average_order_value",
    "annual_spend",
    "web_share",
    "tenure_months",
    "support_contacts_12m",
]
X = customers[features]
truth = customers["segment_truth"]  # Keep out of every fit and selection step.
display(X.describe().round(2))
scaler = StandardScaler()
Z = scaler.fit_transform(X)
print("means:", np.round(Z.mean(axis=0), 3), "stds:", np.round(Z.std(axis=0), 3))

## 1. Centroid

The gradient is $2|C_j|\mu_j-2\sum_{i\in C_j}z_i$; the nonempty-cluster solution is its mean. One point yields zero loss. An empty cluster has no data-determined minimizer and must be reinitialized or handled.

In [ ]:
sample = np.array([-1.0, 0.5, 2.0, 4.0])
mean_center = sample.mean()
grid = np.linspace(-2, 5, 200)
loss = [np.sum((sample - center) ** 2) for center in grid]
plt.plot(grid, loss)
plt.axvline(mean_center, color="black", linestyle="--")
plt.xlabel("candidate center")
plt.ylabel("sum of squared distances")
plt.grid(alpha=0.25)

## 2. Scaling

Large raw scales can dominate distances. Standardization is a weighting choice; inertias in different spaces are not numerically comparable.

In [ ]:
raw = KMeans(n_clusters=4, n_init=20, random_state=606, algorithm="lloyd").fit(X)
scaled = KMeans(n_clusters=4, n_init=20, random_state=606, algorithm="lloyd").fit(Z)
print("raw inertia:", raw.inertia_, "scaled inertia:", scaled.inertia_)
print("feature standard deviations:", X.std().round(2).to_dict())
display(X.assign(cluster=raw.labels_).groupby("cluster").mean().round(2))
display(X.assign(cluster=scaled.labels_).groupby("cluster").mean().round(2))

## 3. PCA/SVD

For $Z=U\Sigma V^T$, $Z^TZ=V\Sigma^2V^T$. Sample covariance eigenvalues are $\sigma^2/(n-1)$. Direction sign is arbitrary.

In [ ]:
centered = Z - Z.mean(axis=0)
_, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
pca_check = PCA(n_components=2, svd_solver="full").fit(centered)
print("absolute alignment:", abs(float(Vt[0] @ pca_check.components_[0])))
print("SVD variance:", singular_values[0] ** 2 / (len(Z) - 1))
print("PCA variance:", pca_check.explained_variance_[0])

## 4. Internal choice and post-hoc ARI

Inertia is monotonic and insufficient by itself; silhouette describes compactness and separation under this geometry. Freeze a candidate before truth comparison.

In [ ]:
rows, fitted = [], {}
for k in range(2, 9):
    estimator = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=606,
        algorithm="lloyd",
    )
    labels = estimator.fit_predict(Z)
    fitted[k] = estimator
    rows.append(
        {
            "k": k,
            "inertia": estimator.inertia_,
            "silhouette": silhouette_score(Z, labels),
        }
    )
diagnostics = pd.DataFrame(rows).set_index("k")
display(diagnostics.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
diagnostics["inertia"].plot(marker="o", ax=axes[0])
axes[0].set(xlabel="k", ylabel="within-cluster sum of squares", title="Inertia")
diagnostics["silhouette"].plot(marker="o", ax=axes[1])
axes[1].set(xlabel="k", ylabel="mean silhouette", title="Internal diagnostic")
plt.tight_layout()

In [ ]:
chosen_k = int(diagnostics["silhouette"].idxmax())
chosen = fitted[chosen_k]
print(
    "k:",
    chosen_k,
    "post-hoc synthetic ARI:",
    adjusted_rand_score(truth, chosen.labels_),
)
repeat = KMeans(
    n_clusters=chosen_k,
    n_init=1,
    init="random",
    random_state=19,
    max_iter=300,
    algorithm="lloyd",
).fit(Z)
print("ARI across starts:", adjusted_rand_score(chosen.labels_, repeat.labels_))
print("inertias:", chosen.inertia_, repeat.inertia_)

## Interpretation

Numeric label IDs can permute without changing a partition. Start stability and synthetic ARI are diagnostics, not proof of a real customer ontology.